In [4]:
import math
import cmath
import sys

# Those are only for Testing
from PIL import Image
import matplotlib.pyplot as plt

# Increase recursion limit for large image dimensions
sys.setrecursionlimit(25000)


# 1. FILE I/O (READ & WRITE PGM FROM SCRATCH) - SUPPORTS P2 & P5


def read_pgm(filename):
    """Reads both ASCII PGM (P2) and Binary PGM (P5) files safely."""
    with open(filename, 'rb') as f:
        data = f.read()

    # Remove comment lines starting with '#' from raw bytes
    cleaned_data = bytearray()
    i = 0
    while i < len(data):
        if data[i:i+1] == b'#':
            while i < len(data) and data[i:i+1] != b'\n':
                i += 1
            i += 1
        else:
            cleaned_data.append(data[i])
            i += 1

    # Extract whitespace-separated tokens to parse headers
    tokens = cleaned_data.split()
    if not tokens:
        raise ValueError("The provided file is empty.")

    magic = tokens[0].decode('latin-1', errors='ignore')
    if magic not in ('P2', 'P5'):
        raise ValueError(f"Unsupported PGM format: {magic}. Only P2 and P5 are supported.")

    width = int(tokens[1].decode('latin-1'))
    height = int(tokens[2].decode('latin-1'))
    max_val = int(tokens[3].decode('latin-1'))

    pixels = []

    if magic == 'P2':
        pixel_data = tokens[4:]
        if len(pixel_data) < width * height:
            raise ValueError("Incomplete pixel data in ASCII PGM.")

        idx = 0
        for _ in range(height):
            row = []
            for _ in range(width):
                row.append(int(pixel_data[idx].decode('latin-1')))
                idx += 1
            pixels.append(row)

    elif magic == 'P5':
        # Binary pixels are exactly the last (width * height) bytes of the file
        raw_pixels = data[-width * height:]
        if len(raw_pixels) < width * height:
            raise ValueError("Incomplete pixel data in Binary PGM.")

        idx = 0
        for _ in range(height):
            row = []
            for _ in range(width):
                row.append(raw_pixels[idx])
                idx += 1
            pixels.append(row)

    return pixels, width, height, max_val

def write_pgm(filename, pixels, width, height, max_val=255):
    """Writes a 2D list of integer pixels into an ASCII PGM file."""
    with open(filename, 'w', encoding='utf-8') as f:
        f.write("P2\n")
        f.write(f"{width} {height}\n")
        f.write(f"{max_val}\n")
        for row in pixels:
            line = " ".join(str(min(max(0, int(round(p))), max_val)) for p in row)
            f.write(line + "\n")



# 2. FFT UTILITIES & MATRIX ZERO-PADDING


def next_power_of_two(n):
    return 1 << (n - 1).bit_length() if n > 0 else 1

def pad_image_to_power_of_two(pixels, width, height):
    new_w = next_power_of_two(width)
    new_h = next_power_of_two(height)

    padded = [[0.0] * new_w for _ in range(new_h)]
    for y in range(height):
        for x in range(width):
            padded[y][x] = pixels[y][x]

    return padded, new_w, new_h

def fft_1d(x):
    N = len(x)
    if N <= 1:
        return x
    even = fft_1d(x[0::2])
    odd = fft_1d(x[1::2])
    T = [cmath.exp(-2j * cmath.pi * k / N) * odd[k] for k in range(N // 2)]
    return [even[k] + T[k] for k in range(N // 2)] + [even[k] - T[k] for k in range(N // 2)]

def ifft_1d(x):
    N = len(x)
    x_conj = [c.conjugate() for c in x]
    X = fft_1d(x_conj)
    return [c.conjugate() / N for c in X]

def fft_2d(matrix):
    height = len(matrix)
    width = len(matrix[0])

    row_fft = [fft_1d(row) for row in matrix]

    col_fft = []
    for x in range(width):
        col = [row_fft[y][x] for y in range(height)]
        col_transformed = fft_1d(col)
        col_fft.append(col_transformed)

    final_fft = [[col_fft[x][y] for x in range(width)] for y in range(height)]
    return final_fft

def ifft_2d(matrix):
    height = len(matrix)
    width = len(matrix[0])

    row_ifft = [ifft_1d(row) for row in matrix]

    col_ifft = []
    for x in range(width):
        col = [row_ifft[y][x] for y in range(height)]
        col_transformed = ifft_1d(col)
        col_ifft.append(col_transformed)

    final_ifft = [[col_ifft[x][y] for x in range(width)] for y in range(height)]
    return final_ifft



# 3. FILTER #2: FREQUENCY HIGH-FREQUENCY EMPHASIS (HFE) FILTER


def apply_hfe_filter(fft_matrix, d0=45, a=1.0, b=0.8, n=2):
    """
    Applies Butterworth High-Frequency Emphasis filter to centered spectrum.
    Parameters tuned for crisp edge sharpening without harsh artifacts:
    a = 1.0 (Retains 100% of the original background lighting)
    b = 0.8 (Clean, moderate edge sharpening boost)
    """
    height = len(fft_matrix)
    width = len(fft_matrix[0])
    y_center = height / 2.0
    x_center = width / 2.0

    filtered = [[0j] * width for _ in range(height)]

    for u in range(height):
        for v in range(width):
            dist = math.sqrt((u - y_center)**2 + (v - x_center)**2)

            if dist == 0:
                h_hp = 0.0
            else:
                h_hp = 1.0 / (1.0 + (d0 / dist) ** (2 * n))

            h_hfe = a + b * h_hp
            filtered[u][v] = fft_matrix[u][v] * h_hfe

    return filtered




In [5]:

# 4. FIXED PIPELINE RUNNER WITH DIRECT VALUE CLAMPING



def run_crisp_image_enhancement(input_filename, output_filename, d0=45, a=1.0, b=0.8):
    print("--- Starting Clean Pipeline ---")

    # 1. Read input PGM
    print("[+] Reading input PGM file...")
    pixels, width, height, max_val = read_pgm(input_filename)
    print(f"Original Dimensions: {width} x {height}")

    # 2. Zero-pad dimensions to power of 2 for FFT
    print("[+] Padding matrix for FFT processing...")
    padded_img, pad_w, pad_h = pad_image_to_power_of_two(pixels, width, height)

    # 3. Center spectrum & run 2D FFT
    print("[+] Centering spectrum and computing 2D FFT...")
    complex_input = []
    for y in range(pad_h):
        row = []
        for x in range(pad_w):
            val = padded_img[y][x] * ((-1) ** (x + y))
            row.append(complex(val, 0.0))
        complex_input.append(row)

    spectrum = fft_2d(complex_input)

    # 4. Frequency filtering (High-Frequency Emphasis)
    print(f"[+] Applying HFE Filter (Cutoff D0={d0}, a={a}, b={b})...")
    filtered_spectrum = apply_hfe_filter(spectrum, d0=d0, a=a, b=b, n=2)

    # 5. Convert back to spatial domain via 2D IFFT
    print("[+] Computing 2D Inverse FFT...")
    ifft_result = ifft_2d(filtered_spectrum)

    # 6. Uncenter using (-1)^(x+y) and extract real part
    uncentered_spatial = [[0.0] * pad_w for _ in range(pad_h)]
    for y in range(pad_h):
        for x in range(pad_w):
            val = ifft_result[y][x].real * ((-1) ** (x + y))
            uncentered_spatial[y][x] = val

    # 7. Crop back to original dimensions
    print("[+] Removing zero-padding...")
    cropped_spatial = [[0.0] * width for _ in range(height)]
    for y in range(height):
        for x in range(width):
            cropped_spatial[y][x] = uncentered_spatial[y][x]

    # 8. DIRECT CLAMPING - No more linear min-max scaling!
    print("[+] Applying pixel clamping to keep true original contrast...")
    final_pixels = []
    for row in cropped_spatial:
        clamped_row = []
        for p in row:
            # Round and clamp strictly to 0 - 255
            val = int(round(p))
            val = max(0, min(255, val))
            clamped_row.append(val)
        final_pixels.append(clamped_row)

    # 9. Export output file
    print(f"[+] Writing processed output to {output_filename}")
    write_pgm(output_filename, final_pixels, width, height, max_val=255)
    print("--- Pipeline Processing Complete ---")


In [6]:

# 5. VISUALIZATION AND EXECUTION CODE


def test_pipeline():
    input_path = "/content/download-_3_.pgm"
    output_path = "/content/my_true_enhanced.pgm"

    try:
        # Run our true-contrast pipeline
        run_crisp_image_enhancement(
            input_filename=input_path,
            output_filename=output_path,
            d0=45,
            a=1.0,
            b=0.8 # Excellent enhancement strength without halos
        )

        # Display side-by-side comparison using matplotlib
        print("\n[+] Rendering comparison...")
        fig, axes = plt.subplots(1, 2, figsize=(12, 6))

        # Plot Original
        img_in = Image.open(input_path)
        axes[0].imshow(img_in, cmap='gray')
        axes[0].set_title(f"Original Input Image")
        axes[0].axis('off')

        # Plot Enhanced
        img_out = Image.open(output_path)
        axes[1].imshow(img_out, cmap='gray')
        axes[1].set_title(f"True Crisp Enhancement")
        axes[1].axis('off')

        plt.tight_layout()
        plt.show()

    except FileNotFoundError:
        print(f"\n[!] File error: Please make sure '{input_path}' is uploaded to Colab.")
    except Exception as e:
        print(f"\n[!] Error encountered: {e}")

# Run the fixed test
test_pipeline()

--- Starting Clean Pipeline ---
[+] Reading input PGM file...

[!] File error: Please make sure '/content/download-_3_.pgm' is uploaded to Colab.


In [7]:

# =====================================================================
# 5. VISUALIZATION AND EXECUTION CODE
# =====================================================================

def test_pipeline(input_path,output_path):
    try:
        # Run our true-contrast pipeline
        run_crisp_image_enhancement(
            input_filename=input_path,
            output_filename=output_path,
            d0=45,
            a=1.0,
            b=0.8 # Excellent enhancement strength without halos
        )

        # Display side-by-side comparison using matplotlib
        print("\n[+] Rendering comparison...")
        fig, axes = plt.subplots(1, 2, figsize=(12, 6))

        # Plot Original
        img_in = Image.open(input_path)
        axes[0].imshow(img_in, cmap='gray')
        axes[0].set_title(f"Original Input Image")
        axes[0].axis('off')

        # Plot Enhanced
        img_out = Image.open(output_path)
        axes[1].imshow(img_out, cmap='gray')
        axes[1].set_title(f"True Crisp Enhancement")
        axes[1].axis('off')

        plt.tight_layout()
        plt.show()

    except FileNotFoundError:
        print(f"\n[!] File error: Please make sure '{input_path}' is uploaded to Colab.")
    except Exception as e:
        print(f"\n[!] Error encountered: {e}")


In [8]:
input_path="/content/gray.pgm"
output_path="/content/grayenhanced.pgm"
test_pipeline(input_path,output_path)

--- Starting Clean Pipeline ---
[+] Reading input PGM file...

[!] File error: Please make sure '/content/gray.pgm' is uploaded to Colab.


In [9]:
input_path="/content/images-_1_.pgm"
output_path=input_path.replace(".pgm", "") + "enhanced.pgm"
test_pipeline(input_path,output_path)

--- Starting Clean Pipeline ---
[+] Reading input PGM file...

[!] File error: Please make sure '/content/images-_1_.pgm' is uploaded to Colab.


In [10]:
input_path="/content/sunflower-grayscale.pgm"
output_path=input_path.replace(".pgm", "") + "enhanced.pgm"
test_pipeline(input_path,output_path)


--- Starting Clean Pipeline ---
[+] Reading input PGM file...

[!] File error: Please make sure '/content/sunflower-grayscale.pgm' is uploaded to Colab.


In [11]:
input_path="/content/grayscale-image-api.pgm"
output_path=input_path.replace(".pgm", "") + "enhanced.pgm"
test_pipeline(input_path,output_path)

--- Starting Clean Pipeline ---
[+] Reading input PGM file...

[!] File error: Please make sure '/content/grayscale-image-api.pgm' is uploaded to Colab.


# Flask Swagger API
This adds a Swagger UI (OpenAPI) for a file-upload endpoint that returns the enhanced PGM. Run locally, then open /apidocs.

In [ ]:
import os
import tempfile
from flask import Flask, request, send_file, jsonify
from flasgger import Swagger

app = Flask(__name__)

swagger_template = {
    "swagger": "2.0",
    "info": {
        "title": "PGM Enhancement API",
        "description": "Upload a PGM image and receive the enhanced result.",
        "version": "1.0.0",
    },
}

swagger_config = {
    "headers": [],
    "specs": [
        {
            "endpoint": "apispec_1",
            "route": "/apispec_1.json",
            "rule_filter": lambda rule: True,
            "model_filter": lambda tag: True,
        }
    ],
    "static_url_path": "/flasgger_static",
    "swagger_ui": True,
    "specs_route": "/apidocs/",
}

Swagger(app, template=swagger_template, config=swagger_config)

@app.post("/api/enhance")
def enhance_api():
    """
    Upload a PGM image and return the enhanced output.
    ---
    consumes:
      - multipart/form-data
    parameters:
      - name: file
        in: formData
        type: file
        required: true
        description: PGM file (P2 or P5).
      - name: d0
        in: formData
        type: number
        required: false
        default: 45
      - name: a
        in: formData
        type: number
        required: false
        default: 1.0
      - name: b
        in: formData
        type: number
        required: false
        default: 0.8
    responses:
      200:
        description: Enhanced PGM file
    """
    if "file" not in request.files:
        return jsonify(error="Missing file"), 400

    uploaded = request.files["file"]
    if uploaded.filename == "":
        return jsonify(error="Empty filename"), 400

    try:
        d0 = float(request.form.get("d0", 45))
        a = float(request.form.get("a", 1.0))
        b = float(request.form.get("b", 0.8))
    except ValueError:
        return jsonify(error="Invalid numeric parameters"), 400

    with tempfile.TemporaryDirectory() as tmp_dir:
        in_path = os.path.join(tmp_dir, "input.pgm")
        out_path = os.path.join(tmp_dir, "output.pgm")
        uploaded.save(in_path)

        run_crisp_image_enhancement(
            input_filename=in_path,
            output_filename=out_path,
            d0=d0,
            a=a,
            b=b,
        )

        return send_file(
            out_path,
            mimetype="image/x-portable-graymap",
            as_attachment=True,
            download_name="enhanced.pgm",
        )

# Run locally with: app.run(host="127.0.0.1", port=5000, debug=True)
